# Lesson 3.8c — 任务条件、语言，以及这个 pool 到底迫使模型读什么

3.8a 定义了契约，3.8b 量了可观测性。本节回答同一串问题：

$$\text{task condition} \;\neq\; \text{language} \quad\Rightarrow\quad \text{什么数据才会迫使模型去读语言？}$$

| 小节 | 问的问题 |
|---|---|
| **3.8.4.1** | `task_goal`（condition）与指令（language）分别提供什么信息 |
| **3.8.4.2** | 为什么当前数据"学不会"语言——以及"学不会"与"不可测"的区别 |
| **3.8.4.3** | 加了第二个任务够不够？实测：观测相似、动作不同，但 `task_goal` 把任务交了出去 |
| **3.8.4.4** | 语言表示从任务编号到 token embedding；为什么两个任务下 L0 与 L1 信息等价 |
| **3.8.4.5** | grounding 的两种形态；任务差异住在哪个通道；以及一张**修正过的**泄漏地图 |

对应 `docs/roadmap_v3.md` 的 3.8.4（前半）。

## 运行说明

本 notebook 覆盖 **3.8.4.1 – 3.8.4.5**。

- 数据契约的实现在 `scripts/mml_contract.py`（**单一来源**）。下面这个 preamble cell 是**唯一**的前置：
  它把契约读进来并暴露 `datasets` / `episodes` / `pick` / `push` / `T_common` / `INSTRUCTIONS` / `VOCAB` /
  `LANGUAGE_IDS` / `build_sample` 等名字。3.8a-3.8d 都 import 同一份实现，所以字段布局与词表不会在
  几本 notebook 之间各自漂移——这是拆分之后最容易出的错。
- 执行顺序：`Kernel → Restart Kernel and Run All Cells`。
- 一个容易踩的 Jupyter 陷阱：**notebook 里显示的输出不一定属于当前 kernel。** 从磁盘重新加载
  notebook 时旧输出仍然显示，但 kernel 是空的——"看起来跑过了"和"状态还在"是两件事。

**命名约定**：`pick` / `push` 是**episode 列表**（`paired_episode_lists()` 给出）；
任务级字典写作 `datasets["PickCube-v1"]`。同一个名字不在同一本里兼指两件事。

In [1]:
# 前置：数据契约来自 scripts/mml_contract.py —— 单一实现。本 notebook 只 import，不重复声明。
import logging
import sys
import warnings
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import mml_contract as mmc

logging.getLogger("mani_skill").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*NVML.*")
warnings.filterwarnings("ignore", message=".*CUDA initialization.*")

datasets = mmc.load_datasets()
episodes = datasets["PickCube-v1"]["episodes"]
pick, push, T_common = mmc.paired_episode_lists(datasets)

INSTRUCTIONS, VOCAB, T_TXT, PAD = mmc.INSTRUCTIONS, mmc.VOCAB, mmc.T_TXT, mmc.PAD
LANGUAGE_IDS = mmc.LANGUAGE_IDS
ACTION_DIM, H = mmc.ACTION_DIM, mmc.H
IMAGE_HW, CHANNELS = mmc.IMAGE_HW, mmc.CHANNELS
PROPRIO_SLICES = datasets["PickCube-v1"]["proprio_slices"]
TASK_GOAL_SLICE = datasets["PickCube-v1"]["goal_slice"]
EXCLUDED_SLICES = datasets["PickCube-v1"]["excluded"]
OBS_DIM = datasets["PickCube-v1"]["state_dim"]
attrs = datasets["PickCube-v1"]["attrs"]
RGB_H5 = datasets["PickCube-v1"]["path"]

tokenize = mmc.tokenize
build_sample = mmc.build_sample
proprio_of = mmc.proprio_of
instruction_variants = mmc.instruction_variants

print(f"contract loaded from {Path(mmc.__file__).name}: {len(datasets)} tasks, "
      f"{sum(len(d['episodes']) for d in datasets.values())} episodes, T_common = {T_common}")

contract loaded from mml_contract.py: 2 tasks, 10 episodes, T_common = 50


## 3.8.4.1 Task condition vs language：symbolic goal 与 semantic instruction

`goal_pos` 与 instruction **同属 condition 层**（3.8.3 的三层框架），但它们不是同一种东西：

| | **symbolic goal**（`task_goal[3]`） | **semantic instruction**（`language_ids`） |
|---|---|---|
| 形式 | 3 个实数 | 一个 token 序列 |
| 谁来解释 | **不需要解释**，直接可用 | 必须被**解释**（哪句话指哪个目标） |
| 作用域 | **每条 episode 一个** | 每个**任务**一个 |
| 精度 | 精确（浮点） | 取决于词汇——"left" 有多左？ |
| 表达力 | 只能说"哪个点" | 还能说"用哪种方式、哪个物体、什么约束" |

一句话：**symbolic goal 是"值"，semantic instruction 是"要求"。**

而本课的现状是：**instruction 目前还没有承担 goal 的能力。** 下一个 cell 把它测出来。

In [2]:
# 3.8.4.1：指令在【任务内】是常量，而 goal 在【任务内】变化 —— 所以指令不能替代 goal

print(f"{'task':>14} {'instruction':<26} {'goal x span':>12} {'goal y span':>12} {'goal z span':>12}")
print("-" * 84)
goals_of = {}
for env_id, ds in datasets.items():
    g = next(x for x in ds["schema"] if x["field"] == "extra.goal_pos")
    goals = np.array([ep["all_state"][0][g["start"]:g["stop"]] for ep in ds["episodes"]])
    goals_of[env_id] = goals
    span = np.ptp(goals, axis=0)
    print(f"{env_id:>14} {INSTRUCTIONS[env_id]:<26} {span[0]:>12.4f} {span[1]:>12.4f} {span[2]:>12.4f}")

print()
for env_id, goals in goals_of.items():
    farthest = max(np.linalg.norm(a - b) for a in goals for b in goals)
    print(f"{env_id}:")
    print(f"  {len(goals)} episodes share ONE instruction  ->  instruction variance within the task = 0")
    print(f"  but goal_pos varies, max pairwise distance = {farthest:.4f} m")

n_tasks = len(datasets)
print()
print(f"across tasks: {n_tasks} distinct instructions -> the channel can carry "
      f"{np.log2(n_tasks):.1f} bit of task identity")

# The two properties this section rests on, asserted rather than assumed.
assert len({INSTRUCTIONS[e] for e in datasets}) == n_tasks, "tasks do not have distinct instructions"
for env_id, goals in goals_of.items():
    assert np.ptp(goals, axis=0).max() > 0, f"{env_id}: goal is constant within the task"

print()
print("=> the instruction is constant within a task and the goal is not, so goal cannot be")
print("   recovered from the instruction, and the instruction cannot substitute for task_goal.")
print("   In this data the instruction carries TASK IDENTITY only.")
print()
print("note: the symbolic goal's dimensionality is itself task dependent -- PushCube's")
print("      goal_z is constant, so its goal lives on the table plane, while PickCube's")
print("      varies over 0.2452 m in z.")

          task instruction                 goal x span  goal y span  goal z span
------------------------------------------------------------------------------------
   PickCube-v1 pick up the cube                 0.1522       0.1192       0.2452
   PushCube-v1 push the cube to the goal        0.1507       0.1325       0.0000

PickCube-v1:
  5 episodes share ONE instruction  ->  instruction variance within the task = 0
  but goal_pos varies, max pairwise distance = 0.2927 m
PushCube-v1:
  5 episodes share ONE instruction  ->  instruction variance within the task = 0
  but goal_pos varies, max pairwise distance = 0.1651 m

across tasks: 2 distinct instructions -> the channel can carry 1.0 bit of task identity

=> the instruction is constant within a task and the goal is not, so goal cannot be
   recovered from the instruction, and the instruction cannot substitute for task_goal.
   In this data the instruction carries TASK IDENTITY only.

note: the symbolic goal's dimensionality is itse

## 3.8.4.2 当前数据为什么"学不会"语言

一个通道如果**每个样本的取值都相同**，它的信息量是**恰好 0**，不是"很小"。这不是模型的
局限，是**数据的性质**。

把这件事写成三层，其中只有第一层在本数据上成立：

$$H(\ell \mid \text{task}) = 0 \quad\text{（每个任务内指令恒定）}$$

$$\Longrightarrow\quad H(a \mid o, \ell) = H(a \mid o) \quad\text{（加上 }\ell\text{ 不改变任何预测）}$$

$$H(\text{goal} \mid \ell) > 0 \quad\text{（goal 在任务内会变，见 3.8.4.1）}$$

第一层是一个**计数**事实，不需要估计熵：每个任务的 5 条 episode 共享同一句话。下面把它算出来
并断言。

**因此有一个可以事先声明的预测**：本数据上"有 language / 无 language"的消融**必然给出一模一样
的数字**——而且这个相等**什么也不能说明**。它是设计的产物，不是关于语言的证据。

In [3]:
# 3.8.4.2：指令通道的容量 —— 每个任务内只有 1 个取值
from collections import Counter


def entropy_bits(values):
    """Empirical Shannon entropy in bits; 0 exactly when every value is the same.

    The clamp is a conditional, not `max(h, 0.0)`: max() returns the FIRST of equal arguments,
    and -0.0 == 0.0, so max(-0.0, 0.0) is -0.0 and still prints as "-0.000", which reads as a
    negative entropy.
    """
    n = len(values)
    counts = Counter(values)
    h = -sum((c / n) * np.log2(c / n) for c in counts.values())
    return h if h > 0.0 else 0.0


print(f"{'scope':>18} {'values':>7} {'H(l)':>10}   meaning")
print("-" * 74)
for env_id, ds in datasets.items():
    values = [INSTRUCTIONS[env_id]] * len(ds["episodes"])
    per_task = entropy_bits(values)
    print(f"{env_id:>18} {len(set(values)):>7} {per_task:>10.3f}   constant within the task")

pooled = [INSTRUCTIONS[ep["env_id"]] for ds in datasets.values() for ep in ds["episodes"]]
h_pooled = entropy_bits(pooled)
print(f"{'pooled (2 tasks)':>18} {len(set(pooled)):>7} {h_pooled:>10.3f}   task identity only")

# The two facts this section rests on.
for env_id, ds in datasets.items():
    values = [INSTRUCTIONS[env_id]] * len(ds["episodes"])
    assert entropy_bits(values) == 0.0, f"{env_id}: instruction is not constant"
assert h_pooled > 0.0, "the pooled instructions are not distinct"
assert abs(h_pooled - np.log2(len(datasets))) < 1e-12, "pooled entropy is not log2(n_tasks)"

# The language tensor is literally identical for every sample of a task, so restricting
# any function of (o, l) to one task gives a function of o alone.
lang = {env_id: build_sample(ds["episodes"][0], 0)["language_ids"] for env_id, ds in datasets.items()}
for env_id, ds in datasets.items():
    for ep in ds["episodes"]:
        assert np.array_equal(build_sample(ep, 0)["language_ids"], lang[env_id]), env_id
print()
print("asserted: every sample of a task carries an IDENTICAL language tensor")
print(f"          tokens for {list(datasets)[0]!r}: {lang[list(datasets)[0]].tolist()}")
print()
print("=> H(l | task) = 0, so H(a | o, l) = H(a | o) exactly. A with-language / without-language")
print("   ablation on this data must return identical numbers BY CONSTRUCTION, and that")
print("   equality says nothing at all about language.")
print()
print("what would give the channel information:")
print("  (a) several tasks                      -> task identity; 1 bit for two tasks")
print(f"     the pool has this now: H(l) = {h_pooled:.3f} bits over {len(datasets)} tasks")
print("  (b) instructions varying WITHIN a task -> goal / parameter information")
print("     the data does NOT have this: H(l | task) = 0 everywhere")

             scope  values       H(l)   meaning
--------------------------------------------------------------------------
       PickCube-v1       1      0.000   constant within the task
       PushCube-v1       1      0.000   constant within the task
  pooled (2 tasks)       2      1.000   task identity only

asserted: every sample of a task carries an IDENTICAL language tensor
          tokens for 'PickCube-v1': [1, 3, 4, 5, 6, 2, 0, 0]

=> H(l | task) = 0, so H(a | o, l) = H(a | o) exactly. A with-language / without-language
   ablation on this data must return identical numbers BY CONSTRUCTION, and that
   equality says nothing at all about language.

what would give the channel information:
  (a) several tasks                      -> task identity; 1 bit for two tasks
     the pool has this now: H(l) = 1.000 bits over 2 tasks
  (b) instructions varying WITHIN a task -> goal / parameter information
     the data does NOT have this: H(l | task) = 0 everywhere


### 读法：区分"学不会"与"不可测"

| 说法 | 是否准确 |
|---|---|
| "模型学不会语言" | ❌ 听着像模型的缺陷，其实不是 |
| "语言通道在这份数据上**没有信息**" | ✅ 计数事实，已断言 |
| "语言通道的贡献**不可测**" | ✅ 最准确的表述 |

"贡献很小"和"贡献不可测"是两件不同的事：前者意味着你还能测量它、只是量小；后者意味着
**任何消融都会返回零，而零不代表它在别处无用**。

顺带一个方法论后果：**如果你在这份数据上做了"加 language"的消融，你得到的零结果是设计决定的，
不是发现的。** 也就是说它**不能作为阴性证据**去支持"语言对策略没用"这个更一般的结论。

### 那什么才能让语言获得信息

| 途径 | 得到什么 | 现在的数据 |
|---|---|---|
| **(a) 多个任务** | 任务身份（两个任务 = 1 bit） | ✅ 已有（PickCube + PushCube） |
| **(b) 任务内指令变化** | goal / 参数信息 | ❌ 没有（每个任务内 H(ℓ)=0） |

3.8.4.4 要定义的 **task-ID 对照（Level 0）** 正是 (a) 这条途径的上限：它给模型的就是那
1 bit。而 (b) 需要"同一任务、不同目标"的指令——那正是 3.8.4.3 要回答"要不要造"的问题。

## 3.8.4.3 最小多任务数据：什么数据才迫使模型读语言

3.8.4.2 的结论是：本数据里语言只承载**任务身份**。这一步问的是——**怎样的任务设计才让那
1 bit 真正被需要？**

判据还是那条（第四次使用）：**同一 observation、不同指令、而必须不同 action。**

但这次要加一个**否定条件**，否则任务设计会失效：

| 两个任务之间的关系 | 模型能否绕过语言 | 是否迫使 |
|---|---|---|
| 画面不同、动作不同 | **能** —— 从图像就能分辨是哪个任务 | ❌ |
| 画面相同、动作相同（同一任务换说法） | 无所谓 —— 语言是装饰 | ❌ |
| **画面相同、动作不同** | **不能** —— 只有指令能分辨 | ✅ |

所以设计要求是：**任务要"看起来像、做起来不像"。**

PickCube 与 PushCube 是按这个要求选的（grasp-and-place vs push），下一个 cell 把它测出来——
**并且会暴露一个同类的混杂**。

In [4]:
# 3.8.4.3：两个任务是否"看起来像、做起来不像"
import itertools

pick = datasets["PickCube-v1"]["episodes"]
push = datasets["PushCube-v1"]["episodes"]
T_common = min(min(len(e["action"]) for e in pick), min(len(e["action"]) for e in push))


def mean_abs_pixel(a, b):
    return float(np.abs(a.astype(np.int16) - b.astype(np.int16)).mean())


def collect(t0, t1):
    img_x, img_w, act_x, act_w = [], [], [], []
    for t in range(t0, t1):
        for e1 in pick:
            for e2 in push:
                img_x.append(mean_abs_pixel(e1["image"][t], e2["image"][t]))
                act_x.append(float(np.linalg.norm(e1["action"][t] - e2["action"][t])))
        for e1, e2 in itertools.combinations(pick, 2):
            img_w.append(mean_abs_pixel(e1["image"][t], e2["image"][t]))
            act_w.append(float(np.linalg.norm(e1["action"][t] - e2["action"][t])))
    return tuple(np.mean(v) for v in (img_x, img_w, act_x, act_w))


print(f"matched time index, t in [0, {T_common - 1}], all episode pairs")
print(f"{'window':>10} {'img cross':>10} {'img within':>11} {'img ratio':>10} "
      f"{'act cross':>10} {'act within':>11} {'act ratio':>10}")
print("-" * 80)
windows = [(0, 10), (10, 20), (20, 30), (30, T_common)]
rows = {}
for t0, t1 in windows:
    ic, iw, ac, aw = collect(t0, t1)
    rows[(t0, t1)] = (ic, iw, ac, aw)
    print(f"{f'[{t0},{t1})':>10} {ic:>10.2f} {iw:>11.2f} {ic / iw:>9.2f}x "
          f"{ac:>10.4f} {aw:>11.4f} {ac / aw:>9.2f}x")

ic, iw, ac, aw = collect(0, T_common)
print("-" * 80)
print(f"{'ALL':>10} {ic:>10.2f} {iw:>11.2f} {ic / iw:>9.2f}x {ac:>10.4f} {aw:>11.4f} {ac / aw:>9.2f}x")

# Property 1: the images do not separate the two tasks.
assert ic / iw < 1.5, f"images separate the tasks too well: ratio {ic / iw:.2f}"
# Property 2: the required actions do, and they do so from the very first steps.
assert ac / aw > 2.0, f"actions barely differ across tasks: ratio {ac / aw:.2f}"
assert rows[(0, 10)][2] / rows[(0, 10)][3] > 10.0, "the tasks are not separable in the first steps"
print()
print("=> observationally similar (image ratio < 1.5) but actionally different")
print(f"   (action ratio {ac / aw:.2f}x overall, {rows[(0, 10)][2] / rows[(0, 10)][3]:.2f}x in the first ten steps)")

# ---- the confound this exposes ------------------------------------------------
g = next(x for x in datasets["PickCube-v1"]["schema"] if x["field"] == "extra.goal_pos")
goal_z = {}
for env_id, ds in datasets.items():
    gg = next(x for x in ds["schema"] if x["field"] == "extra.goal_pos")
    goal_z[env_id] = np.array([ep["all_state"][0][gg["start"]:gg["stop"]][2] for ep in ds["episodes"]])
print()
print("but task_goal gives the task away:")
for env_id, z in goal_z.items():
    print(f"  {env_id:>14} goal_z = {np.round(z, 4)}")
thr = 0.02
correct = sum((z > thr).sum() + (z <= thr).sum() for z in goal_z.values())
n = sum(len(z) for z in goal_z.values())
pick_above = (goal_z["PickCube-v1"] > thr).sum()
push_below = (goal_z["PushCube-v1"] <= thr).sum()
print(f"  a single threshold goal_z > {thr} classifies the task with "
      f"{pick_above + push_below} / {n} episodes correct")
assert pick_above == len(goal_z["PickCube-v1"]) and push_below == len(goal_z["PushCube-v1"]), \
    "goal_z does not separate the tasks"
print()
print("=> so with task_goal in the contract, task identity is readable from the goal's z")
print("   component alone: the two-task pool does NOT force the model to read the instruction.")

matched time index, t in [0, 49], all episode pairs
    window  img cross  img within  img ratio  act cross  act within  act ratio
--------------------------------------------------------------------------------
    [0,10)       6.01        4.77      1.26x     2.0026      0.1005     19.93x
   [10,20)       7.39        5.07      1.46x     2.0194      0.3229      6.25x
   [20,30)       6.88        5.95      1.16x     2.0398      0.4374      4.66x
   [30,50)       6.89        5.95      1.16x     1.0441      0.8659      1.21x
--------------------------------------------------------------------------------
       ALL       6.81        5.54      1.23x     1.6300      0.5185      3.14x

=> observationally similar (image ratio < 1.5) but actionally different
   (action ratio 3.14x overall, 19.93x in the first ten steps)

but task_goal gives the task away:
     PickCube-v1 goal_z = [0.2889 0.2463 0.0487 0.2431 0.2939]
     PushCube-v1 goal_z = [0.001 0.001 0.001 0.001 0.001]
  a single threshol

### 结论：加了第二个任务还不够

| 检查 | 结果 | 含义 |
|---|---|---|
| 画面能否分辨任务 | 跨任务 6.81 vs 同任务 5.54（**1.23×**） | ❌ 不能 |
| 动作能否分辨任务 | 跨任务 1.6300 vs 同任务 0.5185（**3.14×**，前 10 步 **19.93×**） | ✅ 能，且开局就分叉 |
| **`task_goal` 能否分辨任务** | `goal_z > 0.02` → **10/10 正确** | ⚠️ **能**——这就是混杂 |

> **`<补正 3.8.4.5>`** 上面"动作跨任务 3.14×"这个数几乎全部来自**夹爪一个通道**：7 个 arm 通道的
> 比值只有 0.72–1.12×，夹爪是 **7.00×**。所以更准确的说法是"**两个任务在夹爪行为上不同，而手臂
> 轨迹在统计上相似**"。结论不变，但差异的位置变窄了。

> **诚实的结论：这两个任务在"看起来像、做起来不像"这一条上成立，但它们在 `task_goal`
> 上完全可分（PushCube 的 `goal_z ≡ 0.001`，PickCube 在 0.0487–0.2939）。所以在契约含
> `task_goal` 的前提下，模型从目标高度就知道自己在哪个任务——语言仍然不是必需的。**

这和 3.8.3 那次是同一个陷阱的第三次出现（**goal 与 cube 初始位置相关 → goal 泄露场景**；
现在是 **goal 泄露任务**）：**只要存在一个更便宜的通道能解释标签，语言就不会被使用。**

**要真正迫使模型读语言，必须做下面二者之一：**

| 方案 | 做法 | 代价 |
|---|---|---|
| **(a) 让 `task_goal` 在两个任务间不可分辨** | 两个任务从**同一个目标分布**里采样 | 需要改任务的目标采样，或设计一对目标分布相同的任务 |
| **(b) 把 `task_goal` 从输入里拿掉** | 语言成为目标的**唯一**来源 | 需要**指令携带 goal**（合成或标注），即 3.8.4.4 的那条路 |

**本课的选择**：3.8.4.4 会**同时定义 Level 0（task-ID）与 Level 1（token embedding）**，并把
这个混杂**写成显式的对照臂**——即分别测 `image+proprio+task_goal`、`+task-ID`、`+language`
三种输入。这样"语言到底有没有被用"就由**消融**回答，而不是由我们希望。"

### 边界

- **只有 2 个任务** ⇒ 任务身份最多 1 bit。真正的多任务（10+ 个任务、开放指令）是 P1 的题目，
  本课不做。
- `goal_z > 0.02` 这个分离是**这一批数据**的性质（10 条、每任务 5 条），不是任务族的保证。
  换 seed 范围要重新测。

## 3.8.4.4 语言表示：从任务编号到 token embedding

语言的表示分三层，区别不在"维度大小"，而在**是否共享结构**：

| 层级 | 张量 | 指令之间的关系 | 能泛化到新指令吗 |
|---|---|---|---|
| **Level 0 — task ID** | one-hot，`[n_tasks]` | **毫无关系**（互相正交） | ❌ 闭集专用 |
| **Level 1 — learned token embedding** | `[T_txt]` token ids → embedding | 共享词（`the`、`cube`）**共享向量** | 部分（新组合可以） |
| Level 2 — pretrained encoder | 预训练句向量 | 语义邻近性**先验已知** | ✅ 但那是 3.8.7 的事 |

### 关键：在这份数据上，Level 0 与 Level 1 **信息等价**

因为指令与任务是**双射**（3.8.4.1 断言了每个任务内指令恒定，3.8.4.2 断言了两个任务指令不同）：

$$\ell \;\longleftrightarrow\; \text{task}$$

所以 `task_id → language_ids` 与 `language_ids → task_id` 都是确定映射，两个通道对标签携带的
信息**完全相同**。于是：

> **Level 0 是 Level 1 的天花板。** 两者之间的任何性能差都只关于**好不好学**（优化与架构），
> 不关于**能表达什么**（信息）。

这正是本节最该记住的一条：**只有 2 个任务时，"语言"与"任务编号"没有信息上的区别。** 选 token
的唯一理由是它能扩展到更多任务与开放词表——而那是以后的事。

### 三条消融臂（3.8.4.3 的混杂必须显式化）

| 臂 | 输入 | 它回答 |
|---|---|---|
| **A**（现状） | `image + proprio + task_goal` | 完全不读语言能到多少 |
| **B**（Level 0 控制） | `image + proprio + task-ID` | 给 1 bit 任务身份能到多少 —— **这是语言的天花板** |
| **C**（Level 1） | `image + proprio + language` | 语言表示能到多少；与 B 的差只关于可学性 |

In [5]:
# 3.8.4.4：三种语言张量，以及 Level 0 ≡ Level 1 的断言

TASK_IDS = {env_id: np.eye(len(datasets), dtype=np.float32)[i]
            for i, env_id in enumerate(sorted(datasets))}

print(f"vocabulary, {len(VOCAB)} entries: {VOCAB}")
print(f"T_txt = {T_TXT}  (<bos> + longest instruction + <eos>, shorter ones right-padded)")
print()
print(f"{'task':>14} {'arm B: task-ID':>16} {'arm C: language_ids':>24}  tokens")
print("-" * 88)
for env_id in sorted(datasets):
    ids = LANGUAGE_IDS[env_id]
    words = [k for k, v in VOCAB.items() if k not in ("<pad>", "<bos>", "<eos>") and v in ids.tolist()]
    print(f"{env_id:>14} {str(TASK_IDS[env_id].tolist()):>16} {str(ids.tolist()):>24}  {words}")

# Level 0 and Level 1 are informationally equivalent: the pairing is a bijection.
pairs = [(tuple(TASK_IDS[e].tolist()), tuple(LANGUAGE_IDS[e].tolist())) for e in sorted(datasets)]
assert len(set(pairs)) == len(pairs), "task-ID and language are not in bijection"
assert len({tuple(LANGUAGE_IDS[e].tolist()) for e in datasets}) == len(datasets)
assert len({tuple(TASK_IDS[e].tolist()) for e in datasets}) == len(datasets)
print()
print("asserted: task-ID <-> language_ids is a bijection over the pool's tasks")
print("  => H(task | task-ID) = H(task | language) = 0, so arm B caps arm C")

words_of = lambda e: set(INSTRUCTIONS[e].split())
a, b = sorted(datasets)
print()
print(f"shared words     : {sorted(words_of(a) & words_of(b))}")
print(f"only in {a}: {sorted(words_of(a) - words_of(b))}")
print(f"only in {b}: {sorted(words_of(b) - words_of(a))}")
print()
print("the two instructions share 'the cube' and differ in the verb and its arguments, so this")
print("is already a near-minimal pair. A bag-of-words model separates them, but it must use the")
print("verb; the language load here is exactly one bit, which arm B already carries.")

vocabulary, 10 entries: {'<pad>': 0, '<bos>': 1, '<eos>': 2, 'pick': 3, 'up': 4, 'the': 5, 'cube': 6, 'push': 7, 'to': 8, 'goal': 9}
T_txt = 8  (<bos> + longest instruction + <eos>, shorter ones right-padded)

          task   arm B: task-ID      arm C: language_ids  tokens
----------------------------------------------------------------------------------------
   PickCube-v1       [1.0, 0.0] [1, 3, 4, 5, 6, 2, 0, 0]  ['pick', 'up', 'the', 'cube']
   PushCube-v1       [0.0, 1.0] [1, 7, 5, 6, 8, 5, 9, 2]  ['the', 'cube', 'push', 'to', 'goal']

asserted: task-ID <-> language_ids is a bijection over the pool's tasks
  => H(task | task-ID) = H(task | language) = 0, so arm B caps arm C

shared words     : ['cube', 'the']
only in PickCube-v1: ['pick', 'up']
only in PushCube-v1: ['goal', 'push', 'to']

the two instructions share 'the cube' and differ in the verb and its arguments, so this
is already a near-minimal pair. A bag-of-words model separates them, but it must use the
verb; the langua

### 读法与边界

**读法。** `T_txt` 由**最长**指令决定，短的补 `<pad>`——这是必须显式的决定：不补就组不成 batch，
补了就必须带 mask，否则模型会把填充当内容。三条臂的判读规则：

| 观察 | 结论 |
|---|---|
| C ≈ B | 语言张量确实承载了那 1 bit，token 化没有损失信息 |
| C < B | 损失发生在**优化/架构**（词表太小、pooling 方式），不是信息 |
| C > B | **不可能**——出现就说明实验有 bug（B 是上界） |

**边界（必须写清）：**

1. **只有 2 个任务 ⇒ 语言最多 1 bit。** 本节任何结论都**不能**外推成"语言对策略无用"。
2. **Level 2（pretrained encoder）不做**：它会引入语义先验，把"数据够不够"和"预训练强不强"
   混在一起。留给 3.8.7 接 VLA。
3. **`task_goal` 仍在契约里**，所以臂 A 依然会赢或至少不输——3.8.4.3 的混杂**没有消除，只是被
   显式化了**。真正消除需要 (a) 两任务同分布目标，或 (b) 让指令携带 goal，两者都不在本课范围。
4. 词表是**闭合的**（10 个条目、空白切词）。真实 tokenizer 会改变词表大小但**不改变契约的
   shape**：`[T_txt]` 依旧是 `[T_txt]`。

## 3.8.4.5 Language grounding：两种形态，难度差一个量级

"grounding"常被当成一件事，其实是两件，而且**难度不同**：

| | **指称 grounding**（referential） | **动作模式 grounding**（action-mode） |
|---|---|---|
| 词指向什么 | **画面里的一个区域/属性** | **一个行为模式** |
| 例子 | "red cube" ↔ 那个红物体；"left" ↔ 左边那块像素 | "pick" ↔ 抓取并搬运；"push" ↔ 贴桌推 |
| 是否存在"对应的像素" | 有，可以定位 | **没有**——没有一个区域"是 pick" |
| 谁能做 | 需要**视觉-语言绑定**（cross-attention 或预训练对齐） | 只需把语言当作**模式选择器** |
| 本数据能测吗 | ❌ 见下 | ✅ **也只能测这个** |

**为什么本数据测不了指称 grounding：**

1. 两条指令里**没有任何指称词**（没有颜色、没有方位）——`"pick up the cube"` / `"push the cube to
   the goal"` 的全部实词是 `{pick, up, the, cube, push, to, goal}`；
2. 每个场景里**只有一个物体**，所以即使加了 `"red cube"` 也没有别的东西可供区分。

也就是说：**指称 grounding 需要的是场景变化（多个物体、多种属性），不是模型变化。** 这是数据
设计的要求，下一代数据才谈得上。

**机制上它落在哪里。** 在我们 3.8.5 要写的**最小融合模型**里，语言是通过**拼接**进 MLP 的：

$$\text{action} = f\big(\underbrace{\phi_{\text{img}}(I_t)}_{\text{视觉特征}},\ \underbrace{p_t}_{\text{proprio}},\ \underbrace{e(\ell)}_{\text{语言嵌入}}\big)$$

指称 grounding 需要**两个条件同时成立**，而拼接式融合**两个都不满足**：

| | 要求 | 3.8.5 的拼接式模型 |
|---|---|---|
| (a) 视觉保留 **token 网格** | $Z_I \in \mathbb{R}^{n_{\text{patch}} \times d}$，如 $16\times16=256$ 个 patch token | ❌ `φ_img` 把 patch **池化**成一个向量 |
| (b) 有一个 attention 步让语言去**调制**读哪些视觉 token | cross-attention 或 joint self-attention | ❌ 只有拼接 + MLP |

(a) 不是修辞要求。若 $Z_I \in \mathbb{R}^{1 \times d}$，则 `K`、`V` 各只有一个 token：

$$\operatorname{softmax}\!\Big(\frac{QK^\top}{\sqrt{d_k}}\Big) \in \mathbb{R}^{1 \times 1} \equiv [1]$$

"选哪个区域"这个问题**已经没有区域可选**——语言拿到的永远是那个唯一 token 的加权和。

(b) 里"`Q` 来自语言、`K/V` 来自视觉"只是**一种** routing，完整的有三种：

| # | `Q` 来自 | `K`/`V` 来自 | 谁这么干 | 回答的问题 |
|---|---|---|---|---|
| 1 | **语言 token** | 视觉 token | captioning、DETR、VLM connector | "哪个 patch 是 red?" |
| 2 | **全部 token** | 全部 token | OpenVLA、π₀ 的 VLM backbone（joint self-attention） | 所有 token 两两交换 |
| 3 | **action / latent token** | VLM prefix（语言 + 视觉） | π₀ 的 **action expert**、带条件的 diffusion policy | "给定这个场景 + 指令，action 是什么?" |

**#1 做绑定，#3 做控制，两者通常在不同层**——π₀ 正是分层共存（backbone 做 #2、action expert 做
#3）。所以"语言怎么进 policy"至少要问两次：**哪里做绑定、哪里做控制**。这些属于 3.8.7 接 VLA 的事。

**另外，attention 图不等于 grounding。** #1 会产出一张 $A \in \mathbb{R}^{n_{\text{lang}} \times n_{\text{patch}}}$
的 word × patch 矩阵；`"red"` 那一行尖锐**看起来**像"它找到了红色区域"，但行和为 1 只说明它分配了
预算，不说明分配对了。唯一站得住的检验是**反事实**（3.8.4.6）。

**所以本节能做的是：把"动作模式 grounding"量化。** 下一个 cell 问一个具体问题——**任务差异到底
住在动作空间的哪里？**

In [6]:
# 3.8.4.5：任务差异住在动作空间的哪里，以及哪个输入通道泄露了任务
import itertools

# (1) 指称 grounding 需要指称词，也需要可被区分的场景
REFERENTIAL = {"red", "blue", "left", "right", "near", "far", "front", "behind",
               "above", "below", "big", "small"}
content = sorted({w for e in datasets for w in INSTRUCTIONS[e].split()})
print("content words :", content)
print("referential   :", sorted(set(content) & REFERENTIAL))
assert not (set(content) & REFERENTIAL), "the pool does contain referential words; revisit"
print("  -> no colour, no spatial relation, and one object per scene, so referential grounding")
print("     is untestable here by construction, not by model choice.")

# (2) 任务差异落在哪个动作通道上
names = [f"arm{j + 1}" for j in range(7)] + ["gripper"]
print()
print(f"{'channel':>9} {'cross':>9} {'within':>9} {'ratio':>8}")
ratios = []
for j in range(ACTION_DIM):
    cross = [abs(e1["action"][t][j] - e2["action"][t][j])
             for t in range(T_common) for e1 in pick for e2 in push]
    within = [abs(e1["action"][t][j] - e2["action"][t][j])
              for t in range(T_common) for e1, e2 in itertools.combinations(pick, 2)]
    mx, mw = float(np.mean(cross)), float(np.mean(within))
    ratios.append(mx / mw)
    print(f"{names[j]:>9} {mx:>9.4f} {mw:>9.4f} {mx / mw:>7.2f}x")
heavy = [names[j] for j, r in enumerate(ratios) if r > 2.0]
print()
print(f"channels carrying the task difference (ratio > 2): {len(heavy)} / {ACTION_DIM} -> {heavy}")
assert heavy == ["gripper"], f"expected only the gripper, got {heavy}"
print("  -> the difference is LOCALISED, not a global mode change: the arm trajectories are")
print("     statistically similar across tasks; what differs is the gripper's behaviour.")



content words : ['cube', 'goal', 'pick', 'push', 'the', 'to', 'up']
referential   : []
  -> no colour, no spatial relation, and one object per scene, so referential grounding
     is untestable here by construction, not by model choice.

  channel     cross    within    ratio
     arm1    0.0470    0.0501    0.94x
     arm2    0.1123    0.1084    1.04x
     arm3    0.0360    0.0493    0.73x
     arm4    0.1899    0.1977    0.96x
     arm5    0.0363    0.0458    0.79x
     arm6    0.0987    0.0878    1.12x
     arm7    0.1205    0.1669    0.72x
  gripper    1.5120    0.2160    7.00x

channels carrying the task difference (ratio > 2): 1 / 8 -> ['gripper']
  -> the difference is LOCALISED, not a global mode change: the arm trajectories are
     statistically similar across tasks; what differs is the gripper's behaviour.


In [7]:
# (3) 哪个输入通道泄露任务。判据是 achievable accuracy，不是 range overlap。
print()
print("which input channel leaks the task?  criterion: achievable accuracy, NOT range overlap")


def dim_acc(a, b):
    """Best single-threshold accuracy, over BOTH label orientations and all legal splits.

    Two traps, each of which produced a wrong number in the first reading of 3.8.4.5:
      * thresholds taken from the data are a knife edge -- the boundary sample can land on the
        wrong side, so place them strictly BETWEEN neighbouring values;
      * trying only one orientation (a > t) silently tests a hypothesis that may be false by
        construction. Mean brightness is HIGHER for PushCube, so the one-sided version reports
        chance (50.9%) where the two-sided answer is 88.7%.
    """
    a = np.asarray(a, np.float64).ravel()
    b = np.asarray(b, np.float64).ravel()
    vals = np.sort(np.unique(np.concatenate([a, b])))
    if len(vals) == 1:
        return 0.5
    thr = (vals[:-1] + vals[1:]) / 2.0
    best = max(max((a > t).sum() + (b <= t).sum(), (b > t).sum() + (a <= t).sum()) for t in thr)
    return best / (len(a) + len(b))


# image: the cheapest possible statistic, and it is NOT at chance
brightness = {env_id: np.concatenate(
    [e["image"].astype(np.float32).reshape(len(e["image"]), -1).mean(1)
     for e in datasets[env_id]["episodes"]]) for env_id in datasets}
img_acc = dim_acc(brightness["PickCube-v1"], brightness["PushCube-v1"])
print(f"  image (mean brightness) : {100 * img_acc:.1f}% over the whole window   (chance 50%)")
assert img_acc > 0.8, "expected the image to separate the tasks -- 3.8.4.5 claimed the opposite"
print(f"                            PushCube is simply brighter "
      f"({brightness['PushCube-v1'].mean():.3f} vs {brightness['PickCube-v1'].mean():.3f})")

# and t=0 is NOT a clean frame for an image-conditioned policy
print()
print("  does the image leak at t=0, where the proprioception is identical?")
for t in (0, 1, 5):
    acc = dim_acc(np.array([e["image"][t].astype(np.float32).mean() for e in pick]),
                  np.array([e["image"][t].astype(np.float32).mean() for e in push]))
    print(f"    t={t}: image brightness accuracy {100 * acc:>5.1f}%")
img0_equal = all(np.array_equal(pick[i]["image"][0], push[i]["image"][0]) for i in range(len(pick)))
print(f"    t=0 images episode-wise identical across tasks? {img0_equal}")
print("  -> the camera SEES the goal marker, and the two tasks put it in different places, so")
print("     the image names the task from the very first frame.")
assert not img0_equal, "t=0 images are identical; the t=0 clean-frame argument would revive"




which input channel leaks the task?  criterion: achievable accuracy, NOT range overlap
  image (mean brightness) : 88.7% over the whole window   (chance 50%)
                            PushCube is simply brighter (124.091 vs 123.521)

  does the image leak at t=0, where the proprioception is identical?
    t=0: image brightness accuracy 100.0%
    t=1: image brightness accuracy 100.0%
    t=5: image brightness accuracy 100.0%
    t=0 images episode-wise identical across tasks? False
  -> the camera SEES the goal marker, and the two tasks put it in different places, so
     the image names the task from the very first frame.


In [8]:
# proprio: field-aware, dim by dim. The slice differs per task, so it comes from the episode.
PROPRIO_FIELDS = ("agent.qpos", "agent.qvel", "extra.tcp_pose")
q_lo, u_q_lo = pick[0]["proprio_slices"][0][0], push[0]["proprio_slices"][0][0]
t_lo, u_t_lo = pick[0]["proprio_slices"][2][0], push[0]["proprio_slices"][2][0]
print()
print(f"  {'field':>12} {'dim':>4} {'acc':>7} {'range overlap':>14}")
leaky = []
for k, field in enumerate(PROPRIO_FIELDS):
    p_lo, p_hi = pick[0]["proprio_slices"][k]
    u_lo, u_hi = push[0]["proprio_slices"][k]
    A = np.concatenate([e["all_state"][:T_common, p_lo:p_hi] for e in pick], 0)
    B = np.concatenate([e["all_state"][:T_common, u_lo:u_hi] for e in push], 0)
    assert A.shape[1] == B.shape[1], (field, A.shape, B.shape)
    for j in range(A.shape[1]):
        acc = dim_acc(A[:, j], B[:, j])
        ov = max(0.0, min(A[:, j].max(), B[:, j].max()) - max(A[:, j].min(), B[:, j].min()))
        if acc > 0.9:
            leaky.append(f"{field.split('.')[-1]}[{j}]")
            print(f"  {field.split('.')[-1]:>12} {j:>4} {100 * acc:>6.1f}% {ov:>14.5f}   <== separable")
print(f"  single dims above 90%: {leaky}")
assert leaky, "no proprio dim separates the tasks; revisit this conclusion"

# overlap and accuracy disagree, and NOT in one direction
q7 = (np.concatenate([e["all_state"][:T_common, q_lo + 7] for e in pick]),
      np.concatenate([e["all_state"][:T_common, u_q_lo + 7] for e in push]))
tp4 = (np.concatenate([e["all_state"][:T_common, t_lo + 4] for e in pick]),
       np.concatenate([e["all_state"][:T_common, u_t_lo + 4] for e in push]))
print()
for tag, (a, b) in (("qpos[7]", q7), ("tcp_pose[4]", tp4)):
    ov = max(0.0, min(a.max(), b.max()) - max(a.min(), b.min()))
    print(f"  {tag:>12}: range overlap {ov:.5f}  ->  accuracy {100 * dim_acc(a, b):.1f}%")
print("  -> qpos[7] overlaps 0.0217 and is 98% separable; qvel[8] overlaps 0.3045 and is also 98%;")
print("     tcp_pose[4] barely overlaps and lands at 81%. A criterion that is sometimes right and")
print("     sometimes off by 47 points is worse than one that is always wrong: you cannot tell")
print("     which case you are in. 3.8.4.5 read qpos[7:9] from its overlap as 'not separable'.")

# proprio's leak is TEMPORAL; the image's is not
print()
print("  when does proprio become informative?")
for t in (0, 1, 2, 5, T_common - 1):
    a = np.array([e["all_state"][t, q_lo + 7] for e in pick], np.float64)
    b = np.array([e["all_state"][t, u_q_lo + 7] for e in push], np.float64)
    d = max(float(np.abs(
        np.concatenate([pick[i]["all_state"][t][lo:hi] for lo, hi in pick[i]["proprio_slices"]]) -
        np.concatenate([push[i]["all_state"][t][lo:hi] for lo, hi in push[i]["proprio_slices"]])).max())
        for i in range(len(pick)))
    print(f"    t={t:>2}: qpos[7] accuracy {100 * dim_acc(a, b):>5.1f}%   "
          f"max |pick state - push state| = {d:.6f}")
print("  -> proprio is null at t=0 and informative from t=1. The image is informative at t=0.")
assert dim_acc(*q7) > 0.9, "qpos[7] should separate the tasks over the whole window"
assert dim_acc(np.array([e["all_state"][1, q_lo + 7] for e in pick], np.float64),
               np.array([e["all_state"][1, u_q_lo + 7] for e in push], np.float64)) > 0.9
assert dim_acc(np.array([e["all_state"][0, q_lo + 7] for e in pick], np.float64),
               np.array([e["all_state"][0, u_q_lo + 7] for e in push], np.float64)) == 0.5, \
    "t=0 proprio should be exactly chance"
assert all(np.array_equal(
    np.concatenate([pick[i]["all_state"][0][lo:hi] for lo, hi in pick[i]["proprio_slices"]]),
    np.concatenate([push[i]["all_state"][0][lo:hi] for lo, hi in push[i]["proprio_slices"]]),
) for i in range(len(pick))), "t=0 proprio is not episode-wise identical"
print()
print("=> EVERY channel in the contract leaks the task:")
print("     image     88.7% over the window, 100% at t=0   (the camera sees the goal marker)")
print("     task_goal 10/10 episodes                      (goal_z separates completely)")
print("     proprio   98.0% from t=1                      (qpos[7] is the gripper)")
print("   No frame and no field subset makes the instruction necessary for an image-conditioned")
print("   policy on this pool. 3.8.4.6 has to pre-register that, and test language on a")
print("   deliberately restricted probe instead.")



         field  dim     acc  range overlap
          qpos    7   98.0%        0.02171   <== separable
          qpos    8   98.0%        0.02173   <== separable
          qvel    8   98.0%        0.30450   <== separable


  single dims above 90%: ['qpos[7]', 'qpos[8]', 'qvel[8]']



       qpos[7]: range overlap 0.02171  ->  accuracy 98.0%
   tcp_pose[4]: range overlap 0.00050  ->  accuracy 81.4%
  -> qpos[7] overlaps 0.0217 and is 98% separable; qvel[8] overlaps 0.3045 and is also 98%;
     tcp_pose[4] barely overlaps and lands at 81%. A criterion that is sometimes right and
     sometimes off by 47 points is worse than one that is always wrong: you cannot tell
     which case you are in. 3.8.4.5 read qpos[7:9] from its overlap as 'not separable'.

  when does proprio become informative?
    t= 0: qpos[7] accuracy  50.0%   max |pick state - push state| = 0.000000
    t= 1: qpos[7] accuracy 100.0%   max |pick state - push state| = 0.304505
    t= 2: qpos[7] accuracy 100.0%   max |pick state - push state| = 0.184862
    t= 5: qpos[7] accuracy 100.0%   max |pick state - push state| = 0.103887
    t=49: qpos[7] accuracy 100.0%   max |pick state - push state| = 0.694767
  -> proprio is null at t=0 and informative from t=1. The image is informative at t=0.

=> EVERY 

### 读法，以及对 3.8.4.3 的一处修正

**任务差异的位置**（实测）：7 个 arm 通道的跨任务/同任务比只有 **0.72–1.12×**，夹爪是 **7.00×**。

> **`<补正 3.8.4.3>`** 那一节说"动作跨任务差 3.14×（前 10 步 19.93×）"——数值没错，但**差异
> 几乎全部来自夹爪一个通道**。所以更准确的说法是：**两个任务在夹爪行为上不同，而手臂轨迹在统计上
> 相似**。结论（动作向量确实不同、画面确实不能分辨）不变，但差异的**位置**不是我当时暗示的"整条
> 手臂"。这也让"语言测试"变窄了一点：模型可以靠"指令 → 夹爪时序"通过，而不必真的把动词绑定到
> 别的什么上面。

**`<补正 3.8.4.5>` 我原来的泄漏地图是错的。** 我当时用 **range overlap** 当可分性判据：

| 维度 | 区间重叠 | 我原来的结论 | **真实单阈值准确率** |
|---|---|---|---|
| `qpos[7]`（手指） | 0.02171（大） | "不可分" ❌ | **98.0%** |
| `qvel[8]` | 0.30450（很大） | （没测） | **98.0%** |
| `tcp_pose[4]` | 0.00051（极小） | "近乎可分" | **81.4%** —— 这条大致对了 |

**overlap 是 range（极值）统计量，可分性是 achievable accuracy（分布）统计量。** 问题不在于它
总是高估或总是低估：`qpos[7]` 重叠 2% 却 98% 可分，`qvel[8]` 重叠 0.30 也是 98%，而
`tcp_pose[4]` 恰好估得差不多。**一个有时对、有时错得离谱的判据比一个一直错的判据更危险**，
因为你看不出自己落在哪种情况——这也是为什么"用 overlap 判断可分性"必须从方法上删掉，而不是
调个阈值了事。修正后的地图：

| 通道 | 每帧能读出任务吗 | 证据 |
|---|---|---|
| `image` | ✅ **能** | 平均亮度单阈值 **88.7%**（整段），**`t=0` 起就 100%** |
| `proprio` | ✅ **能** | `qpos[7]` 单维 **98.0%**；`proprio(25)` 线性探针（leave-one-episode-out）**96.3%** |
| `task_goal` | ✅ 完全 | `goal_z > 0.02` → 10/10 |

**`image` 这一行我也错了**：原来是 `❌ 50.9%`。错因是代码只试了**一个标签方向**
（`bi > thr`），而 PushCube 比 PickCube **更亮**，所以那个方向测的是一个"从构造上就为假"的假设；
两个方向都试，答案是 **88.7%**。（这是本节的第三个错，也是最有教育意义的一个：**只试一个方向
等于偷偷假设了结论。**）

`proprio` 的泄漏**是时序的**：`t=0` 时两个任务的 proprio **逐 episode 完全相同**（差恰为 0），
`qpos[7]` 准确率 **50.0%**；从 `t=1` 起就是 **100%**——因为 PushCube 的 `a_0` 已经把夹爪关下去了。

**所以 3.8.4.3 的 (b) 方案不够，而且比"不够"更糟：**

> ~~只要在语言条件下把 `task_goal` 拿掉，指令就成为每帧唯一能区分两个任务的通道。~~
>
> **不对。** 三个通道**全都**泄露任务：

| 通道 | 整段 | `t=0` |
|---|---|---|
| `image` | 88.7% | **100%** |
| `proprio` | 98.0% | **50.0%** ← 唯一在 `t=0` 干净的一路 |
| `task_goal` | 10/10 | 10/10 |

> 我在 3.8.4.5 里写过"唯一干净的一帧是 `t=0`"。**这句话在 `proprio` 上成立，但对 `image` 不
> 成立**：`t=0` 两个任务的 proprio 逐 episode 完全相同（差恰为 0），可它们的**画面不同**——因为
> 相机看得见**放在桌上的 goal marker**，而两个任务的 goal 位置本来就不同。
>
> **结论：在这个 pool 上，对任何带 `image` 的策略，都不存在"迫使它读语言"的一帧，也不存在这样
> 一个字段子集。** 这不是本 notebook 能修好的事，是**数据设计**问题（P1）。所以 3.8.4.6 必须
> **预注册这个否定结果**，并把语言检验搬到一个刻意受限的 probe 上：
> **`t=0` + 只用 `proprio` + 去掉 `image` 与 `task_goal`**——那时指令是唯一能说出任务的输入。

**边界：**

1. **只有动作模式 grounding 可测。** 指称 grounding 需要多个物体与属性词——那是数据设计的事，
   下一代数据才谈得上。
2. **3.8.5 的拼接式融合两条都缺**：既没有 attention，`φ_img` 又把 spatial token 网格**池化成了
   一个向量**。没有 token 网格，`K`/`V` 各只有一个 token，`softmax` 退化成 `[1]`——"选哪个区域"
   已经没有区域可选。所以语言只能当模式选择器。
3. **"某一帧不可观测" ≠ "策略必须读语言"。** closed loop 里 `t≥1` 的 state 反馈可以自我纠正
   `t=0` 的猜错（夹爪可以重开）。所以终判是 P0 的 **closed-loop success**，不是 offline loss。
4. **夹爪的时序差异正是 memory 的入场点**：`t=0` 的 `proprio` 看不见，带历史的模型看得见。所以
   "3.8 不需要 memory"这个判断，前提是**单帧 + 只做模式选择**；一旦要识别"夹爪是否已经彻底闭合"
   这类 *过程性* 状态，历史就变成必要条件。
5. **"画面能分辨任务"与"画面有任务信号"是两件事。** 88.7% 来自一个全局亮度偏移（PushCube 更亮），
   它足以让模型**抄近路**，却不含任何有语义的物体信息——3.8.3 已经量过：goal 在画面里
   `corr = −0.295`，跨 episode 的像素变化与 episode 内 25 步的变化一样大（1.0×）。所以这个 88.7%
   是**混淆**，不是**表征**。

## 小结

1. **task condition 与 language 是两类输入**：`task_goal` 是 *condition*（给定、没有证据来源），
   指令是 *language*（必须被读取）。把两者混在一个"输入"里，是 3.8.4.3 那个混杂的来源。
2. **指令只带 1 bit**：任务内 `H(ℓ) = 0`，跨任务 `H(ℓ) = 1.000` bit，而且这 1 bit 就是任务身份。
3. **有 `task_goal` 在，指令是冗余的**——模型忽略它不受任何惩罚。这是"学不会"与"不可测"的区别：
   本 pool 是**不可测**，不是学不会。
4. **语言表示分三级**：L0 任务编号 / L1 token embedding / L2 预训练编码器（留给 3.8.7）。两个任务下
   L0 与 L1 **信息等价**，所以 L0 是 L1 的**上界**——臂 B 封顶臂 C，**C > B 说明实验坏了**。
5. **任务差异是局域化的**：7 个 arm 通道跨任务/同任务比只有 **0.72–1.12×**，夹爪 **7.00×**。
   所以两个任务在**夹爪行为**上不同，而手臂轨迹在统计上相似。
6. **本 pool 的每个输入通道都泄露任务**：`image` 88.7%（`t=0` 起 **100%**，相机看得见 goal marker）、
   `proprio` 98.0%（`t=1` 起，`qpos[7]` 就是夹爪）、`task_goal` 10/10。
   **没有任何一帧、也没有任何字段子集**能让指令成为 image-conditioned 策略的必需输入——这是**数据设计**问题（P1）。
7. **测量可分性有三个坑，本节的三个错误各踩了一个**：range overlap ≠ achievable accuracy；
   阈值必须取相邻值的**中间点**；必须试**两个**标签方向。

## 自检

1. 为什么"任务内 `H(ℓ) = 0`"就意味着指令在这份数据里**不可能**被迫使用？
2. 两个任务下 L0（任务编号）与 L1（token embedding）为什么信息等价？换成 5 个任务还等价吗？
3. `tcp_pose[4]` 的区间重叠只有 0.0005 却只有 81.4% 可分，而 `qpos[7]` 重叠 0.0217 却有 98.0% 可分。
   这说明 overlap 作为判据的性质是什么？为什么"有时对、有时错"比"一直错"更危险？
4. 指称 grounding 需要**两个条件**，分别是什么？为什么 3.8.5 的拼接式融合两个都不满足？
5. `<补正 3.8.4.3>` 说"动作跨任务差 3.14×"。这个**数字**错了吗？错的是数字还是解释？
6. 如果把 `image` 也从臂 C 里拿掉，只剩 `proprio + language`，那么 `t=0` 和 `t=1` 两种情况下的答案一样吗？